<a href="https://colab.research.google.com/github/suphanatchanlek30/Super-AI-Engineer-Season-6-Mini-Hackathon-Week-4-2-Word-Segmentation/blob/main/Word_Segmentation_600367_%E0%B8%A8%E0%B8%B8%E0%B8%A0%E0%B8%93%E0%B8%B1%E0%B8%90_(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload kaggle.json only

Saving kaggle.json to kaggle (2).json


Cell 2

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!ls -la ~/.kaggle

total 16
drwxr-xr-x 2 root root 4096 Apr  3 18:08 .
drwx------ 1 root root 4096 Apr  3 18:08 ..
-rw------- 1 root root   71 Apr  3 18:20 kaggle.json


Cell 3



In [ ]:
!pip -q install kaggle TorchCRF seqeval

Cell 4

In [ ]:
!kaggle competitions download -c super-ai-engineer-ss-6-word-segmentation -p /content

super-ai-engineer-ss-6-word-segmentation.zip: Skipping, found more recently modified local copy (use --force to force download)


Cell 5

In [ ]:
import zipfile
from pathlib import Path

COMP_ZIP = Path("/content/super-ai-engineer-ss-6-word-segmentation.zip")
COMP_DIR = Path("/content/super-ai-engineer-ss-6-word-segmentation")

COMP_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(COMP_ZIP, "r") as zf:
    zf.extractall(COMP_DIR)

print("COMP_DIR =", COMP_DIR)
print(sorted([p.name for p in COMP_DIR.iterdir()]))


COMP_DIR = /content/super-ai-engineer-ss-6-word-segmentation
['LST20 Annotation Guideline.pdf', 'LST20 Brief Specification.pdf', 'ws_list.txt', 'ws_sample_submission.csv', 'ws_test.txt']


Cell 6

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload opend_lst20_corpus.zip

Saving opend_lst20_corpus.zip to opend_lst20_corpus (2).zip


Cell 7

In [ ]:
from pathlib import Path
import zipfile

def find_lst20_root():
    candidates = [
        Path("/content/opend_lst20_corpus/LST20_Corpus"),
        Path("/content/LST20_Corpus"),
        Path("/content/opend-lst20-corpus/LST20_Corpus"),
    ]
    for c in candidates:
        if c.exists():
            return c
    return None

LST20_DIR = find_lst20_root()
print("before unzip =", LST20_DIR)

if LST20_DIR is None:
    zip_candidates = [
        Path("/content/opend_lst20_corpus.zip"),
        Path("/content/lst20.zip"),
    ]

    chosen_zip = None
    for zp in zip_candidates:
        if zp.exists():
            chosen_zip = zp
            break

    if chosen_zip is None:
        for zp in Path("/content").glob("*.zip"):
            if "lst20" in zp.name.lower() or "opend_lst20_corpus" in zp.name.lower():
                chosen_zip = zp
                break

    print("chosen_zip =", chosen_zip)

    if chosen_zip is not None:
        extract_dir = Path("/content/opend_lst20_corpus")
        extract_dir.mkdir(parents=True, exist_ok=True)

        with zipfile.ZipFile(chosen_zip, "r") as zf:
            zf.extractall(extract_dir)

LST20_DIR = find_lst20_root()
print("after unzip =", LST20_DIR)
assert LST20_DIR is not None, "หา LST20_Corpus ไม่เจอ"


before unzip = /content/opend_lst20_corpus/LST20_Corpus
after unzip = /content/opend_lst20_corpus/LST20_Corpus


Cell 8

In [ ]:
for sub in ["train", "eval", "test"]:
    p = LST20_DIR / sub
    print(sub, p.exists(), len(list(p.glob("*.txt"))) if p.exists() else 0)


train True 3794
eval True 474
test True 483


Cell 9

In [ ]:
import random
import unicodedata
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from TorchCRF import CRF


Cell 10

In [ ]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)
print("cuda available =", torch.cuda.is_available())

if torch.cuda.is_available():
    print("gpu name =", torch.cuda.get_device_name(0))


DEVICE = cpu
cuda available = False


Cell 11

In [ ]:
TEST_PATH = COMP_DIR / "ws_test.txt"
SAMPLE_PATH = COMP_DIR / "ws_sample_submission.csv"
LABEL_LIST_PATH = COMP_DIR / "ws_list.txt"
GENRES_PATH = LST20_DIR / "genres.txt"

print(TEST_PATH.exists(), SAMPLE_PATH.exists(), LABEL_LIST_PATH.exists(), GENRES_PATH.exists())


True True True True


Cell 12

In [ ]:
LABELS = ["B_WORD", "I_WORD", "E_WORD"]
LABEL2ID = {label: idx for idx, label in enumerate(LABELS)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

PAD_LABEL_ID = -100
PAD_CHAR = "<PAD>"
UNK_CHAR = "<UNK>"

print(LABEL2ID)


{'B_WORD': 0, 'I_WORD': 1, 'E_WORD': 2}


Cell 13

In [ ]:
sample_df = pd.read_csv(SAMPLE_PATH)
test_text = TEST_PATH.read_text(encoding="utf-8")

non_ws_count = sum(1 for ch in test_text if not ch.isspace())
ws_count = sum(1 for ch in test_text if ch.isspace())

print("all chars =", len(test_text))
print("non-whitespace chars =", non_ws_count)
print("whitespace chars =", ws_count)
print("sample rows =", len(sample_df))
print(sample_df.head())

assert len(sample_df) == non_ws_count


all chars = 37248
non-whitespace chars = 35182
whitespace chars = 2066
sample rows = 35182
   Id Predicted
0   1    B_WORD
1   2    I_WORD
2   3    E_WORD
3   4       NaN
4   5       NaN


Cell 14

In [ ]:
def read_text(path: Path) -> str:
    return path.read_text(encoding="utf-8")

def load_genre_map(path: Path) -> Dict[str, str]:
    genre_map = {}
    with path.open(encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            doc_name, genre = line.split("\t")
            genre_map[doc_name.replace(".txt", "")] = genre
    return genre_map

genre_map = load_genre_map(GENRES_PATH)
print("num genres =", len(genre_map))
list(genre_map.items())[:5]


num genres = 4751


[('T00126', 'politics'),
 ('T00127', 'C&A'),
 ('T00128', 'C&A'),
 ('T00129', 'general'),
 ('T00130', 'C&A')]

Cell 15

In [ ]:
def token_to_word_labels(token: str) -> List[str]:
    n = len(token)
    if n == 1:
        return ["E_WORD"]
    if n == 2:
        return ["B_WORD", "E_WORD"]
    return ["B_WORD"] + ["I_WORD"] * (n - 2) + ["E_WORD"]


Cell 16

In [ ]:
def parse_lst20_document(path: Path) -> List[Dict]:
    sequences = []
    chars = []
    labels = []

    def flush():
        nonlocal chars, labels, sequences
        if chars:
            sequences.append({
                "doc_id": path.stem,
                "chars": chars,
                "labels": labels,
                "text": "".join(chars),
            })
            chars = []
            labels = []

    with path.open(encoding="utf-8") as f:
        for raw_line in f:
            line = raw_line.rstrip("\n")

            if not line.strip():
                flush()
                continue

            parts = line.split("\t")
            if len(parts) < 4:
                continue

            token = parts[0]

            if token == "_":
                continue

            chars.extend(list(token))
            labels.extend(token_to_word_labels(token))

    flush()
    return sequences


Cell 17

In [ ]:
def load_labeled_sequences(lst20_dir: Path, splits=("train", "eval", "test")) -> List[Dict]:
    all_sequences = []
    for split in splits:
        split_dir = lst20_dir / split
        for path in sorted(split_dir.glob("*.txt")):
            seqs = parse_lst20_document(path)
            for seq in seqs:
                seq["split"] = split
                seq["genre"] = genre_map.get(seq["doc_id"], "unknown")
            all_sequences.extend(seqs)
    return all_sequences

all_sequences = load_labeled_sequences(LST20_DIR, splits=("train", "eval", "test"))
print("num sequences =", len(all_sequences))
print(all_sequences[0]["doc_id"], all_sequences[0]["split"], all_sequences[0]["genre"])
print(all_sequences[0]["text"][:100])
print(all_sequences[0]["labels"][:30])


num sequences = 74180
T00126 train politics
สุรยุทธ์ยันปฏิเสธลงนามMOUกับอียูไม่กระทบสัมพันธ์
['B_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD', 'I_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD', 'E_WORD', 'B_WORD', 'I_WORD']


Cell 18

In [ ]:
def summarize_dataset(sequences: List[Dict]) -> pd.DataFrame:
    label_counter = Counter()
    lengths = []
    docs = set()

    for seq in sequences:
        label_counter.update(seq["labels"])
        lengths.append(len(seq["chars"]))
        docs.add(seq["doc_id"])

    row = {
        "num_docs": len(docs),
        "num_sequences": len(sequences),
        "num_characters": sum(lengths),
        "mean_seq_len": float(np.mean(lengths)),
        "median_seq_len": float(np.median(lengths)),
        "p95_seq_len": float(np.percentile(lengths, 95)),
    }
    for label in LABELS:
        row[f"count_{label}"] = label_counter[label]

    return pd.DataFrame([row])

summarize_dataset(all_sequences)


,num_docs,num_sequences,num_characters,mean_seq_len,median_seq_len,p95_seq_len,count_B_WORD,count_I_WORD,count_E_WORD
0,4751,74180,11523899,155.350485,137.0,353.0,2595469,6241070,2687360


Cell 19



In [ ]:
def build_doc_table(sequences: List[Dict]) -> pd.DataFrame:
    rows = []
    seen = set()

    for seq in sequences:
        if seq["doc_id"] in seen:
            continue
        seen.add(seq["doc_id"])
        rows.append({
            "doc_id": seq["doc_id"],
            "genre": seq["genre"],
            "split": seq["split"],
        })
    return pd.DataFrame(rows)

doc_table = build_doc_table(all_sequences)
doc_table.head()


,doc_id,genre,split
0,T00126,politics,train
1,T00127,C&A,train
2,T00128,C&A,train
3,T00129,general,train
4,T00130,C&A,train


Cell 20

In [ ]:
USE_ALL_LST20_FOR_TRAINING_POOL = False

if USE_ALL_LST20_FOR_TRAINING_POOL:
    pool_sequences = all_sequences
else:
    pool_sequences = [x for x in all_sequences if x["split"] in {"train", "eval"}]

pool_doc_table = build_doc_table(pool_sequences)

train_docs, valid_docs = train_test_split(
    pool_doc_table,
    test_size=0.10,
    random_state=SEED,
    stratify=pool_doc_table["genre"] if pool_doc_table["genre"].nunique() > 1 else None,
)

train_doc_ids = set(train_docs["doc_id"].tolist())
valid_doc_ids = set(valid_docs["doc_id"].tolist())

train_sequences = [x for x in pool_sequences if x["doc_id"] in train_doc_ids]
valid_sequences = [x for x in pool_sequences if x["doc_id"] in valid_doc_ids]

print("train sequences =", len(train_sequences))
print("valid sequences =", len(valid_sequences))


train sequences = 61697
valid sequences = 7233


Cell 21



In [ ]:
def build_vocab(sequences: List[Dict], min_freq: int = 1) -> Dict[str, int]:
    counter = Counter()
    for seq in sequences:
        counter.update(seq["chars"])

    vocab = {PAD_CHAR: 0, UNK_CHAR: 1}
    for ch, freq in counter.items():
        if freq >= min_freq:
            vocab[ch] = len(vocab)
    return vocab

char_vocab = build_vocab(train_sequences)
print("vocab size =", len(char_vocab))
list(char_vocab.items())[:20]


vocab size = 181


[('<PAD>', 0),
 ('<UNK>', 1),
 ('ส', 2),
 ('ุ', 3),
 ('ร', 4),
 ('ย', 5),
 ('ท', 6),
 ('ธ', 7),
 ('์', 8),
 ('ั', 9),
 ('น', 10),
 ('ป', 11),
 ('ฏ', 12),
 ('ิ', 13),
 ('เ', 14),
 ('ล', 15),
 ('ง', 16),
 ('า', 17),
 ('ม', 18),
 ('M', 19)]

Cell 22

In [ ]:
CHAR_TYPE2ID = {
    "PAD": 0,
    "THAI": 1,
    "LATIN": 2,
    "DIGIT": 3,
    "PUNCT": 4,
    "MARK": 5,
    "OTHER": 6,
}

def get_char_type(ch: str) -> int:
    code = ord(ch)
    cat = unicodedata.category(ch)

    if 0x0E00 <= code <= 0x0E7F:
        if cat.startswith("M"):
            return CHAR_TYPE2ID["MARK"]
        return CHAR_TYPE2ID["THAI"]

    if ch.isdigit():
        return CHAR_TYPE2ID["DIGIT"]

    if ("A" <= ch <= "Z") or ("a" <= ch <= "z"):
        return CHAR_TYPE2ID["LATIN"]

    if cat.startswith("P") or cat.startswith("S"):
        return CHAR_TYPE2ID["PUNCT"]

    if cat.startswith("M"):
        return CHAR_TYPE2ID["MARK"]

    return CHAR_TYPE2ID["OTHER"]


Cell 23

In [ ]:
def chunk_sequence(chars: List[str], labels: List[str] = None, max_len: int = 256, overlap: int = 64):
    stride = max_len - overlap
    assert stride > 0

    n = len(chars)
    chunks = []
    start = 0

    while start < n:
        end = min(start + max_len, n)
        chunk = {
            "start": start,
            "end": end,
            "chars": chars[start:end],
        }
        if labels is not None:
            chunk["labels"] = labels[start:end]
        chunks.append(chunk)

        if end == n:
            break
        start += stride

    return chunks


Cell 24

In [ ]:
@dataclass
class Config:
    max_len: int = 256
    overlap: int = 64
    char_emb_dim: int = 128
    type_emb_dim: int = 16
    hidden_dim: int = 256
    lstm_layers: int = 2
    dropout: float = 0.2
    batch_size: int = 64
    lr: float = 1e-3
    weight_decay: float = 1e-5
    epochs: int = 10

CFG = Config()
CFG


Config(max_len=256, overlap=64, char_emb_dim=128, type_emb_dim=16, hidden_dim=256, lstm_layers=2, dropout=0.2, batch_size=64, lr=0.001, weight_decay=1e-05, epochs=10)

Cell 25

In [ ]:
class LST20CharDataset(Dataset):
    def __init__(self, sequences: List[Dict], char_vocab: Dict[str, int], cfg: Config):
        self.samples = []
        self.char_vocab = char_vocab
        self.cfg = cfg

        for seq in sequences:
            chunks = chunk_sequence(
                chars=seq["chars"],
                labels=seq["labels"],
                max_len=cfg.max_len,
                overlap=cfg.overlap,
            )
            for chunk in chunks:
                self.samples.append({
                    "doc_id": seq["doc_id"],
                    "chars": chunk["chars"],
                    "labels": chunk["labels"],
                    "start": chunk["start"],
                    "end": chunk["end"],
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        return {
            "doc_id": item["doc_id"],
            "start": item["start"],
            "end": item["end"],
            "chars": item["chars"],
            "input_ids": [self.char_vocab.get(ch, self.char_vocab[UNK_CHAR]) for ch in item["chars"]],
            "type_ids": [get_char_type(ch) for ch in item["chars"]],
            "labels": [LABEL2ID[x] for x in item["labels"]],
        }

def collate_fn(batch):
    max_len = max(len(x["input_ids"]) for x in batch)
    bs = len(batch)

    input_ids = torch.zeros(bs, max_len, dtype=torch.long)
    type_ids = torch.zeros(bs, max_len, dtype=torch.long)
    labels = torch.full((bs, max_len), PAD_LABEL_ID, dtype=torch.long)
    mask = torch.zeros(bs, max_len, dtype=torch.bool)

    meta = []
    for i, item in enumerate(batch):
        seq_len = len(item["input_ids"])
        input_ids[i, :seq_len] = torch.tensor(item["input_ids"], dtype=torch.long)
        type_ids[i, :seq_len] = torch.tensor(item["type_ids"], dtype=torch.long)
        labels[i, :seq_len] = torch.tensor(item["labels"], dtype=torch.long)
        mask[i, :seq_len] = True
        meta.append({
            "doc_id": item["doc_id"],
            "start": item["start"],
            "end": item["end"],
            "chars": item["chars"],
        })

    return {
        "input_ids": input_ids,
        "type_ids": type_ids,
        "labels": labels,
        "mask": mask,
        "meta": meta,
    }


Cell 26

In [ ]:
USE_PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 2 if torch.cuda.is_available() else 0

train_dataset = LST20CharDataset(train_sequences, char_vocab, CFG)
valid_dataset = LST20CharDataset(valid_sequences, char_vocab, CFG)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG.batch_size,
    shuffle=True,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=USE_PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG.batch_size,
    shuffle=False,
    collate_fn=collate_fn,
    num_workers=NUM_WORKERS,
    pin_memory=USE_PIN_MEMORY,
    persistent_workers=(NUM_WORKERS > 0),
)

print("train chunks =", len(train_dataset))
print("valid chunks =", len(valid_dataset))
print("pin_memory =", USE_PIN_MEMORY)
print("num_workers =", NUM_WORKERS)
print("batch_size =", CFG.batch_size)
print("num train batches =", len(train_loader))


train chunks = 72635
valid chunks = 8716
pin_memory = False


Cell 27



In [ ]:
class CharBiLSTMCRF(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        char_emb_dim: int,
        type_vocab_size: int,
        type_emb_dim: int,
        hidden_dim: int,
        num_labels: int,
        lstm_layers: int = 2,
        dropout: float = 0.2,
    ):
        super().__init__()

        self.char_emb = nn.Embedding(vocab_size, char_emb_dim, padding_idx=0)
        self.type_emb = nn.Embedding(type_vocab_size, type_emb_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)

        input_dim = char_emb_dim + type_emb_dim

        self.encoder = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim // 2,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
            bidirectional=True,
        )

        self.classifier = nn.Linear(hidden_dim, num_labels)
        self.crf = CRF(num_labels)

    def forward(self, input_ids, type_ids, mask, labels=None):
        x = torch.cat([self.char_emb(input_ids), self.type_emb(type_ids)], dim=-1)
        x = self.dropout(x)

        encoded, _ = self.encoder(x)
        encoded = self.dropout(encoded)

        emissions = self.classifier(encoded)
        mask = mask.bool()

        if labels is not None:
            safe_labels = labels.masked_fill(labels < 0, 0)
            log_likelihood = self.crf(emissions, safe_labels, mask)
            loss = -log_likelihood.mean()
            return {"loss": loss}

        pred_ids = self.crf.viterbi_decode(emissions, mask)
        return {"pred_ids": pred_ids}


Cell 28

In [ ]:
model = CharBiLSTMCRF(
    vocab_size=len(char_vocab),
    char_emb_dim=CFG.char_emb_dim,
    type_vocab_size=len(CHAR_TYPE2ID),
    type_emb_dim=CFG.type_emb_dim,
    hidden_dim=CFG.hidden_dim,
    num_labels=len(LABELS),
    lstm_layers=CFG.lstm_layers,
    dropout=CFG.dropout,
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG.lr,
    weight_decay=CFG.weight_decay,
)

print("num params (M) =", round(sum(p.numel() for p in model.parameters()) / 1e6, 3))


num params (M) = 0.7


Cell 29

In [ ]:
def flatten_valid_labels(labels_tensor, mask_tensor):
    y_true = []
    for labels_row, mask_row in zip(labels_tensor.cpu().numpy(), mask_tensor.cpu().numpy()):
        valid = mask_row.astype(bool)
        y_true.extend(labels_row[valid].tolist())
    return y_true

def evaluate(model, loader):
    model.eval()
    losses = []
    all_true = []
    all_pred = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            type_ids = batch["type_ids"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            mask = batch["mask"].to(DEVICE)

            out = model(input_ids=input_ids, type_ids=type_ids, mask=mask, labels=labels)
            losses.append(out["loss"].item())

            pred_ids = model(input_ids=input_ids, type_ids=type_ids, mask=mask)["pred_ids"]
            batch_true = flatten_valid_labels(labels, mask)

            batch_pred = []
            for seq in pred_ids:
                batch_pred.extend(seq)

            all_true.extend(batch_true)
            all_pred.extend(batch_pred)

    macro_f1 = f1_score(all_true, all_pred, average="macro")
    per_label_f1 = f1_score(all_true, all_pred, average=None, labels=[0, 1, 2])

    metrics = {
        "loss": float(np.mean(losses)),
        "macro_f1": float(macro_f1),
        "f1_B_WORD": float(per_label_f1[0]),
        "f1_I_WORD": float(per_label_f1[1]),
        "f1_E_WORD": float(per_label_f1[2]),
    }
    return metrics, all_true, all_pred


Cell 30



In [ ]:
best_score = -1.0
best_state = None
history = []

num_batches = len(train_loader)

for epoch in range(1, CFG.epochs + 1):
    model.train()
    train_losses = []

    print(f"\n========== Epoch {epoch}/{CFG.epochs} ==========")

    for batch_idx, batch in enumerate(train_loader, start=1):
        input_ids = batch["input_ids"].to(DEVICE)
        type_ids = batch["type_ids"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        mask = batch["mask"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        loss = model(
            input_ids=input_ids,
            type_ids=type_ids,
            mask=mask,
            labels=labels
        )["loss"]

        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        train_losses.append(loss.item())

        # log ทุก 20 batch หรือ batch สุดท้าย
        if batch_idx % 20 == 0 or batch_idx == num_batches:
            avg_train_loss = float(np.mean(train_losses))
            print(
                f"[Epoch {epoch}/{CFG.epochs}] "
                f"Batch {batch_idx}/{num_batches} "
                f"train_loss={loss.item():.4f} "
                f"avg_train_loss={avg_train_loss:.4f}"
            )

    valid_metrics, _, _ = evaluate(model, valid_loader)

    epoch_log = {
        "epoch": epoch,
        "train_loss": float(np.mean(train_losses)),
        **valid_metrics,
    }
    history.append(epoch_log)

    print("\nEpoch Summary")
    print(f"epoch        : {epoch}")
    print(f"train_loss   : {epoch_log['train_loss']:.4f}")
    print(f"valid_loss   : {epoch_log['loss']:.4f}")
    print(f"macro_f1     : {epoch_log['macro_f1']:.4f}")
    print(f"f1_B_WORD    : {epoch_log['f1_B_WORD']:.4f}")
    print(f"f1_I_WORD    : {epoch_log['f1_I_WORD']:.4f}")
    print(f"f1_E_WORD    : {epoch_log['f1_E_WORD']:.4f}")

    if valid_metrics["macro_f1"] > best_score:
        best_score = valid_metrics["macro_f1"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        print(f"new best model saved | best_macro_f1={best_score:.4f}")
    else:
        print(f"no improvement | best_macro_f1={best_score:.4f}")



========== Epoch 1/10 ==========
[Epoch 1/10] Batch 20/1135 train_loss=21.6383 avg_train_loss=29.4539
[Epoch 1/10] Batch 40/1135 train_loss=27.4885 avg_train_loss=28.6124
[Epoch 1/10] Batch 60/1135 train_loss=24.5912 avg_train_loss=27.2858
[Epoch 1/10] Batch 80/1135 train_loss=21.7960 avg_train_loss=26.2731
[Epoch 1/10] Batch 100/1135 train_loss=23.4430 avg_train_loss=25.4407
[Epoch 1/10] Batch 120/1135 train_loss=20.7853 avg_train_loss=24.7840
[Epoch 1/10] Batch 140/1135 train_loss=20.9179 avg_train_loss=24.2639
[Epoch 1/10] Batch 160/1135 train_loss=17.1789 avg_train_loss=23.6966
[Epoch 1/10] Batch 180/1135 train_loss=20.1888 avg_train_loss=23.1108
[Epoch 1/10] Batch 200/1135 train_loss=17.4252 avg_train_loss=22.5805
[Epoch 1/10] Batch 220/1135 train_loss=18.2022 avg_train_loss=22.1157
[Epoch 1/10] Batch 240/1135 train_loss=18.3929 avg_train_loss=21.6987
[Epoch 1/10] Batch 260/1135 train_loss=14.0193 avg_train_loss=21.3012
[Epoch 1/10] Batch 280/1135 train_loss=14.8529 avg_train_los

Cell 31

In [ ]:
history_df = pd.DataFrame(history)
history_df


Cell 32

In [ ]:
if best_state is not None:
    model.load_state_dict(best_state)

valid_metrics, y_true, y_pred = evaluate(model, valid_loader)
print(valid_metrics)


Cell 33

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2])
cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)
cm_df


Cell 34

In [ ]:
def prepare_test_text(path: Path) -> Tuple[str, List[int], str]:
    print(f"[prepare_test_text] reading file: {path}")

    raw_text = read_text(path)
    print(f"[prepare_test_text] loaded text length = {len(raw_text)}")

    nonspace_chars = []
    raw_indices = []

    for idx, ch in enumerate(raw_text):
        if not ch.isspace():
            nonspace_chars.append(ch)
            raw_indices.append(idx)

        if (idx + 1) % 5000 == 0 or (idx + 1) == len(raw_text):
            print(
                f"[prepare_test_text] processed {idx + 1}/{len(raw_text)} chars | "
                f"current non-whitespace = {len(nonspace_chars)}"
            )

    compact_text = "".join(nonspace_chars)

    print("[prepare_test_text] finished")
    print(f"[prepare_test_text] all chars          = {len(raw_text)}")
    print(f"[prepare_test_text] non-whitespace     = {len(compact_text)}")
    print(f"[prepare_test_text] whitespace removed = {len(raw_text) - len(compact_text)}")

    return raw_text, raw_indices, compact_text


raw_test_text, raw_test_indices, compact_test_text = prepare_test_text(TEST_PATH)

print("\nFinal Check")
print("all chars =", len(raw_test_text))
print("non-whitespace chars =", len(compact_test_text))
print("sample rows =", len(sample_df))

assert len(compact_test_text) == len(sample_df)
print("assert passed: test length matches sample submission rows")


Cell 35

In [ ]:
def predict_long_text(model, text: str, char_vocab: Dict[str, int], cfg: Config):
    chars = list(text)
    chunks = chunk_sequence(
        chars=chars,
        labels=None,
        max_len=cfg.max_len,
        overlap=cfg.overlap
    )

    print("[predict_long_text] start")
    print(f"[predict_long_text] total chars  = {len(chars)}")
    print(f"[predict_long_text] total chunks = {len(chunks)}")
    print(f"[predict_long_text] max_len      = {cfg.max_len}")
    print(f"[predict_long_text] overlap      = {cfg.overlap}")

    votes = [Counter() for _ in range(len(chars))]

    model.eval()
    with torch.no_grad():
        for chunk_idx, chunk in enumerate(chunks, start=1):
            input_ids = torch.tensor(
                [[char_vocab.get(ch, char_vocab[UNK_CHAR]) for ch in chunk["chars"]]],
                dtype=torch.long,
                device=DEVICE,
            )
            type_ids = torch.tensor(
                [[get_char_type(ch) for ch in chunk["chars"]]],
                dtype=torch.long,
                device=DEVICE,
            )
            mask = torch.ones_like(input_ids, dtype=torch.bool, device=DEVICE)

            pred_ids = model(
                input_ids=input_ids,
                type_ids=type_ids,
                mask=mask
            )["pred_ids"][0]

            for offset, pred_id in enumerate(pred_ids):
                global_idx = chunk["start"] + offset
                votes[global_idx][pred_id] += 1

            if chunk_idx % 20 == 0 or chunk_idx == 1 or chunk_idx == len(chunks):
                print(
                    f"[predict_long_text] chunk {chunk_idx}/{len(chunks)} "
                    f"| span=({chunk['start']}, {chunk['end']}) "
                    f"| chunk_len={len(chunk['chars'])}"
                )

    print("[predict_long_text] merging votes")

    final_pred_ids = []
    for i, counter in enumerate(votes):
        if not counter:
            final_pred_ids.append(LABEL2ID["E_WORD"])
        else:
            final_pred_ids.append(counter.most_common(1)[0][0])

        if (i + 1) % 5000 == 0 or (i + 1) == len(votes):
            print(f"[predict_long_text] merged {i + 1}/{len(votes)} positions")

    print("[predict_long_text] done")
    return final_pred_ids


Cell 36

In [ ]:
test_pred_ids = predict_long_text(model, compact_test_text, char_vocab, CFG)
test_pred_labels = [ID2LABEL[x] for x in test_pred_ids]

print("num predictions =", len(test_pred_labels))
print(pd.Series(test_pred_labels).value_counts())


Cell 37

In [ ]:
submission_df = sample_df.copy()
submission_df["Predicted"] = test_pred_labels
submission_df.head(10)


Cell 38

In [ ]:
output_path = Path("/content/submission.csv")
submission_df.to_csv(output_path, index=False)
print("saved to", output_path)


Cell 39

In [ ]:
from google.colab import files
files.download("/content/submission.csv")
